In [ ]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")
sys.path.append("../../../")

In [ ]:
experiment_file = '../../experiments/parallelized_experiments/output/square_with_ellipse_hole/2024_01_13_21_03///experiment_result.json'
stiffness_path = '../../experiments/parallelized_experiments/output/square_with_ellipse_hole/2024_01_13_21_03/'
name = 'square_with_ellipse_hole'

In [ ]:
import MeshFEM, visualization

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
m = MeshFEM.mesh.Mesh('../../experiments/parallelized_experiments/output/square_with_ellipse_hole/2024_01_13_21_03/0.60_59.00/mesh_square_with_ellipse_hole_0.60_59.00.obj')
fusing_vtx = np.load('../../experiments/parallelized_experiments/output/square_with_ellipse_hole/2024_01_13_21_03/0.60_59.00/fusedVtx_square_with_ellipse_hole_0.60_59.00.npy')

visualization.plot_2d_mesh(m, pointList=fusing_vtx, width=5, height=5)

In [ ]:
# experiment_file = '../../experiments/parallelized_experiments/output/double_zigzag_dash/2024_01_08_18_15/experiment_result.json'

# stiffness_path = '../../experiments/parallelized_experiments/output/double_zigzag_dash/2024_01_08_18_15'
# name = 'double_zigzag_dash'

### Overview

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [ ]:
df = pd.DataFrame(data['data'])
fig, axes = plt.subplots(nrows = 1, ncols = 3, figsize = (20, 5))
a = (df.hist('Ipu simulation succeed', ax = axes[0]), df.hist('Planar equilibrium', ax = axes[1]), df.hist('Simulation Kappa value', ax = axes[2]))

In [ ]:
import visualize_stiffness
import importlib
importlib.reload(visualize_stiffness)

In [ ]:
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [ ]:
invalid_tags = np.array(df['name'][df['Planar equilibrium'] != 1])

In [ ]:
invalid_tags

In [ ]:
df['Simulation Kappa value'][np.array(df['Planar equilibrium']) != 1]

In [ ]:
kappa_path = None

In [ ]:
def filter_tags(tags, second_number=None, third_number=None):
    filtered_tags = []

    for tag in tags:
        numbers = [float(num) for num in tag.split('_')]

        if second_number is not None and numbers[0] != second_number:
            continue
        if third_number is not None and numbers[1] != third_number:
            continue

        filtered_tags.append(tag)

    return np.array(filtered_tags)

In [ ]:
valid_tags = filter_tags(valid_tags, second_number=1.6)

In [ ]:
radius = np.array(data['pattern_parameters'][0]['values'])
angles = np.array(data['pattern_parameters'][1]['values'])

In [ ]:
radius, angles

In [ ]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [ ]:
(x_scale_factors - y_scale_factors)[np.where(x_scale_factors < y_scale_factors)[0]]

In [ ]:
valid_tags[np.where(x_scale_factors < y_scale_factors)[0]]

In [ ]:
np.max(y_scale_factors), np.argmax(y_scale_factors)

In [ ]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, name, valid_tags, plot_data = False)

In [ ]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [ ]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [ ]:
(x_scale_factors - y_scale_factors)[np.where(x_scale_factors < y_scale_factors)[0]]

In [ ]:
min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

In [ ]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, name, valid_tags)

In [ ]:
np.argmax(min_bending_stiffness)

In [ ]:
np.argmin(min_bending_stiffness)

In [ ]:
len(angles), len(radius)

In [ ]:
angles

In [ ]:
samples = np.array([[float(n) for n in tag.split('_')] for tag in valid_tags])

In [ ]:
plt.scatter(samples[:, 0], samples[:, 1])

In [ ]:
def show_tags(threshold = 0):
    curr_tags = valid_tags[(np.where(min_bending_stiffness > threshold))]
    samples = np.array([[float(n) for n in tag.split('_')] for tag in curr_tags])
    plt.scatter(samples[:, 0], samples[:, 1])

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
radius, angles

In [ ]:
interact(show_tags, threshold=widgets.FloatSlider(min=-1, max=2, step=0.05, value=0));

### Get scale function convex hull

In [ ]:
import matplotlib.cm as cm
import matplotlib as mpl

In [ ]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

In [ ]:
hull

In [ ]:
import matplotlib.pyplot as plt
plt.plot(points[:,0], points[:,1], 'o')
for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')
plt.plot(points[hull.vertices,0], points[hull.vertices,1], 'r--', lw=2)
plt.plot(points[hull.vertices[0],0], points[hull.vertices[0],1], 'ro')
plt.show()

In [ ]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (10, 10))

# plt.scatter(x_scale_factor, y_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)
# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)

# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.8, c = min_stiffness)


points = np.concatenate((x_scale_factors.reshape((-1, 1)), y_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

ax.title.set_text("Scale factors")
plt.xlabel("x scale factors")
plt.ylabel("y scale factors")

plt.scatter(x_scale_factors, y_scale_factors, label = 'min_stiffness', s = 200, alpha = 1, c = min_bending_stiffness)
# plt.scatter(x_scale_factors, y_scale_factors, label = 'max_stiffness', s = 200, alpha = 1, c = max_bending_stiffness)

# Plot x = y line
lims = [
np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
ax.set_aspect('equal')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()
fig.tight_layout()
plt.savefig('scale_factor_values_{}.png'.format(name), dpi = 300)

### Validate the max and min scale factors are aligned with the x and y axis

In [ ]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [ ]:
eqns = hull.equations

In [ ]:
import parametrization_helper, importlib
importlib.reload(parametrization_helper)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, max_scale_factors, min_scale_factors)

### Generate data without augmenting

In [ ]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [ ]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [ ]:
plt.plot(stiffness_coefficients[:, 4])

In [ ]:
np.set_printoptions(suppress=True, precision=4)

In [ ]:
np.argmax(stiffness_coefficients[:, 1]), np.argmax(stiffness_coefficients[:, 2])

In [ ]:
def get_stiffness_polynomial(s, theta):
    return s[0] * np.cos(theta)**2 * np.sin(theta)**2 + s[1] * np.cos(theta)**3 * np.sin(theta) + s[2] * np.cos(theta) * np.sin(theta)**3 + s[3] * np.cos(theta)**4 + s[4] * np.sin(theta)**4

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
# grid_data = np.zeros((9, len(radius), len(angles)))

# for i in range(len(valid_tags)):
#     radius_index = int(i / len(angles))
#     angles_index = int(i % len(angles))

#     grid_data[0][radius_index][angles_index] = max_scale_factors[i]
#     grid_data[1][radius_index][angles_index] = min_scale_factors[i]
#     grid_data[2][radius_index][angles_index] = x_scale_factors[i]
#     grid_data[3][radius_index][angles_index] = y_scale_factors[i]
#     for s in range(5):
#         grid_data[4 + s][radius_index][angles_index] = stiffness_coefficients[i][s]

In [ ]:
grid_data = np.zeros((9, len(angles)))

for i in range(len(valid_tags)):
    grid_data[0][i] = max_scale_factors[i]
    grid_data[1][i] = min_scale_factors[i]
    grid_data[2][i] = x_scale_factors[i]
    grid_data[3][i] = y_scale_factors[i]
    for s in range(5):
        grid_data[4 + s][i] = stiffness_coefficients[i][s]

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
grid_data.shape

In [ ]:
splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, (angles))

In [ ]:
grid_data.shape

In [ ]:
angles

In [ ]:
scale_factors_grid_data = np.zeros((2, len(angles)))
for i in range(len(angles)):
    scale_factors_grid_data[0][i] = x_scale_factors[i]
    scale_factors_grid_data[1][i] = y_scale_factors[i]
scale_factors_splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(scale_factors_grid_data, (angles))

In [ ]:
test_parameters = np.linspace(47, 59, 100)

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
# titles = ['max scale factors', 'min scale factors', 's1', 's2', 's3', 's4', 's5']

for i in range(9):
    axes[i].plot(test_parameters, splines[i * 3 + 0](test_parameters))
    axes[i].set_title(titles[i], fontsize=21)

In [ ]:
stiffness_coefficients = np.array(stiffness_coefficients)

In [ ]:
stiffness_coefficients.shape

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
data = [max_scale_factors, min_scale_factors, x_scale_factors, y_scale_factors, stiffness_coefficients[:, 0], stiffness_coefficients[:, 1], stiffness_coefficients[:, 2], stiffness_coefficients[:, 3], stiffness_coefficients[:, 4]]

for i in range(9):
    axes[i].plot(angles, data[i])
    axes[i].set_title(titles[i], fontsize=21)

### Parametrization

In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
import utils, mesh_utilities
importlib.reload(utils)

In [ ]:
target_surf = mesh.Mesh("../../../../examples/igloo.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
lines = np.array(eqns)

### New local global with convex hull

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.setLines(eqns)
lg.alphaMin = hull.min_bound[0]
lg.alphaMax = hull.max_bound[0]

lg.betaMin = hull.min_bound[1]
lg.betaMax = hull.max_bound[1]

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
lg.alphaMin, lg.alphaMax, lg.betaMin, lg.betaMax

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(lg, show_main = True)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, lg.getAlphas(), lg.getBetas())

### Pattern parameters optimization

In [ ]:
default_pattern_params = np.array([55]  * len(lg.getAlphas()))

In [ ]:
# mat_info = np.array(default_pattern_params).reshape((2, len(lg.getAlphas())))

In [ ]:
default_pattern_params

In [ ]:
angles

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[47, 59]])
rparam.patternParamNormalizationFactors = np.array([59 - 47])
rparam.diffRegW = 0.0

In [ ]:
rparam.patternRegP

In [ ]:
visualization.visualize_both(rparam, height = 4)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [ ]:
rparam.bendRegW = 1

In [ ]:
rparam.energy(PET.RGP)

In [ ]:
rparam.energy(PET.Bending)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
def optimize_rparam(param, patternRegW, phiRegW, bendRegW = 0.0, update_uv = True, niter = 100):
    param.patternRegW = patternRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = niter
    opts.gradTol = 1e-9
    opts.factorizer = opts.factorizer.CatamariNesdis
    benchmark.reset()
    
    if update_uv:
        fixedvars = [param.uOffset(), param.vOffset(), param.phiOffset()]
    else:
        fixedvars = range(param.stretchOffset())

    cr = parametrization.pattern_parametrization_knitro(param, opts.niter, fixedvars)
    benchmark.report()
    return cr

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()


benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, rparam.getAlphas(), rparam.getBetas())

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1)

In [ ]:
rparam.phiRegW = 3e-5
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 3e-5, bendRegW = 0, update_uv = True, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1, width = 30)

In [ ]:
importlib.reload(parametrization_helper)
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
rparam.patternRegW = 5e-5
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 5e-5, phiRegW = 3e-5, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 5e-5, phiRegW = 1e-5, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-5, phiRegW = 1e-6, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1, width = 30)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

### Bending

In [ ]:
rparam.bendRegW = 1e-3
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-5, phiRegW = 1e-6, bendRegW = 1e-3, update_uv = False, niter = 1000)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1, width = 30)

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-5, phiRegW = 1e-6, bendRegW = 1e-3, update_uv = True, niter = 200)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 5e-6, phiRegW = 1e-6, bendRegW = 5e-4, update_uv = True, niter = 200)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-6, phiRegW = 1e-6, bendRegW = 1e-4, update_uv = True, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1, width = 30)

In [ ]:
importlib.reload(parametrization_helper)
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False, width = 5, height = 5)

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_TRI, orientationHue=False, width = 10, height = 10)

## Upsampling and channel generation

In [ ]:
import parametrization_helper
importlib.reload(parametrization_helper)

In [ ]:
def fusing_curve_polyline(patternParams):
#     Draw dash_line
    # r = patternParams[0]
    r = 1.6
    angle = patternParams[0]

    iw = np.cos(angle / 180 * np.pi) * r
    ih = np.sin(angle / 180 * np.pi) * r
    
    
    # Center of the ellipse
    h = k = 0

    # Number of points in the fusing line
    num_points = 20

    # Parameter values
    t = np.linspace(0, 2*np.pi, num_points)
    # Compute the points on the ellipse
    x = h + iw * np.cos(t)
    y = k + ih * np.sin(t)

    # Combine x and y into an array of points
    points = np.column_stack((x, y))
    return [(points + np.array([2.5, 2.5])) / 5 * np.pi]

In [ ]:
points = fusing_curve_polyline([60])

In [ ]:
# fusing_curve_polyline([2, 60])

In [ ]:
fusing_lines = fusing_curve_polyline([60])[0].reshape(-1, 2)
fusing_edges = [[i, i + 1] for i in range(len(fusing_lines) - 1)]

In [ ]:
boundary_vertices = [[0, 0], [np.pi, 0], [np.pi, np.pi], [0, np.pi]]

In [ ]:
boundary_edges = np.array([[0, 1], [1, 2], [2, 3], [3, 0]]) + len(fusing_lines)

In [ ]:
visualization.plot_line_segments(list(fusing_lines) + boundary_vertices, fusing_edges + list(boundary_edges))

In [ ]:
sdfVertices, sdfTris, sdf, sheet_vxs, concatenated_polylines, sheet_edges_polylines, boundaryVxs, boundaryEdges, upsampleMesh_vertices,  upsampleMesh_triangles, upsampledAngles, upsampledPatternParams = parametrization_helper.get_polyline_from_pattern_parameters(rparam, fusing_curve_polyline, nsubdiv = 3, frequency=0.1, duplicates_removable_threshold=[1e-4, 1e-2, 1e-1, 1e0, 2e0, 3e0])

In [ ]:

#### Get raw boundary edges
raw_flatten_mesh = MeshFEM.mesh.Mesh(upsampleMesh_vertices, upsampleMesh_triangles)
raw_boundaryLoop = raw_flatten_mesh.boundaryElements()
raw_boundaryVxsIdxs = raw_flatten_mesh.boundaryVertices()
raw_boundaryVxs = upsampleMesh_vertices[raw_boundaryVxsIdxs]

In [ ]:
if len(boundaryEdges) > 1:
    concatenated_boundary_edges = []
    for polyline in boundaryEdges:
        concatenated_boundary_edges.extend(polyline)
    concatenated_boundary_edges = np.array(concatenated_boundary_edges)
else:
    concatenated_boundary_edges = np.array(boundaryEdges[0])

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 5, height=5)
visualization.plot_line_segments(sheet_vxs, concatenated_polylines, width = 5, height = 5)
visualization.plot_line_segments(list(sheet_vxs) + list(boundaryVxs), list(concatenated_polylines) + list(concatenated_boundary_edges + len(sheet_vxs)), width = 5, height = 5)
plt.scatter(boundaryVxs[concatenated_boundary_edges[:, 0], 0], boundaryVxs[concatenated_boundary_edges[:, 0], 1], c = np.arange(len(boundaryVxs)), cmap = mpl.colormaps['Greys'])

## Meshing and inflation simulation

In [ ]:
import mesher_helper
importlib.reload(mesher_helper)

In [ ]:
import time
time_stamp = time.strftime("%Y_%m_%d_%H_%M")

In [ ]:
selected_elements = [np.array(sublist)[:, 0] for sublist in boundaryEdges[1:]]
holes_vxs = []
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    
    # Check if the polyline forms a closed loop
    if polyline[0, 0] == polyline[-1, 1]:
        # If it's a closed loop, append it to holes_vxs
        holes_vxs.append(np.array(sheet_vxs[polyline[:, 0]]))
#     else:
#         # If it's not a closed loop, find the segment along the boundary that connects the two endpoints
#         start_point = polyline[0, 0]
#         end_point = polyline[-1, 1]
#         boundary_segment = []
#         print(start_point, end_point)
#         for element in selected_elements:
#             if start_point in element and end_point in element:
#                 boundary_segment = element
#                 break

#         # Add the boundary segment to the polyline
#         polyline = np.concatenate((polyline, boundary_segment))

#         # Append the entire closed loop to holes_vxs
#         holes_vxs.append(np.array(list(sheet_vxs[polyline[:, 0]]) + list([sheet_vxs[polyline[-1, 1]]])))

In [ ]:
importlib.reload(mesher_helper)

In [ ]:
v, f, fusing_data = mesher_helper.generate_mesh_non_periodic(4, boundaryVxs[np.array(boundaryEdges[0])[:, 0]], holes_vxs, [], [], gui = False)

In [ ]:
import numpy as np
import copy

# Use the function
new_v, new_f, new_fusing_without_boundary = parametrization_helper.remove_dangling_vertices(v, f - 1, fusing_data)
m = MeshFEM.mesh.Mesh(new_v, new_f)
new_fusing = copy.copy(new_fusing_without_boundary)
new_fusing[m.boundaryVertices()] = True

In [ ]:
fusing_data, new_fusing

In [ ]:
# m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, SV, SE, triArea=1e0)


In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(new_fusing) == 1)[0], width=10, height=10)


In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, new_fusing)

### Save pattern

In [ ]:
polylines = []
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    polylines.append(sheet_vxs[np.array(list(polyline[:, 0]) + list([polyline[-1, 1]]))][:, :2].tolist())
parametrization_helper.save_to_obj(boundaryVxs[np.array(boundaryEdges)[0][:, 0]], polylines, 'igloo_{}_sheet_pattern_{}_margin_{}.obj'.format(name, time_stamp, channelMargin))

### End

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
import boundaries
bdryVars = boundaries.getOuterBoundaryVars(isheet)
fixedVars = bdryVars

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = bdryVars, 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [ ]:
isheet.pressure = 5e-2

In [ ]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [ ]:
import gzip

In [ ]:
pickle.dump(isheet,  gzip.open("igloo_pattern_optimized_{}_low_frequency_with_bending_high_resolution.pkl.gz".format(time_stamp), 'wb'))

### Generate Fabrication Files

In [ ]:
old_to_new = np.arange(np.max(isheet.wallVertices()) + 1)

In [ ]:
old_to_new[isheet.wallVertices()] = np.arange(len(isheet.wallVertices()))

In [ ]:
from parametrization_helper import form_polylines

In [ ]:
result_vxs = isheet.restWallVertexPositions()
result_edges = old_to_new[isheet.wallBoundaryEdges()]
result_edges = form_polylines(result_edges.tolist())
concatenated_polylines = []
for polyline in result_edges:
    concatenated_polylines.extend(polyline)


In [ ]:
visualization.plot_line_segments(result_vxs, concatenated_polylines)